# IEA-15-240-RWT VolturnUS-S — the WISDEM/WindIO one-click pipeline

End-to-end worked example of the **one-click WISDEM/WindIO FOWT
pipeline** added in pyBmodes 1.4.0 (issue #35). A single WindIO
ontology `.yaml` drives:

1. The composite-layup **blade** reduced to a distributed beam by a
   PreComp-class thin-wall multi-cell classical-lamination analysis
   (`RotatingBlade.from_windio`).
2. The tubular **tower** straight from the ontology geometry
   (`Tower.from_windio`).
3. The coupled **floating platform**, two-tier by design
   (`Tower.from_windio_floating`): an explicitly-labelled screening
   preview from the yaml alone, and the BModes-JJ-validated
   industry-grade coupled model once the companion
   HydroDyn/MoorDyn/ElastoDyn decks are present.
4. The **one-click orchestrator** itself (`pybmodes windio`), which
   auto-discovers those companion decks scoped to the turbine root.
5. A **Campbell** sweep and a **MAC** cross-check, with
   engineering-paper-styled plots via `pybmodes.plots`.

Run end-to-end with **Kernel → Restart & Run All**; total wall time
~ 40 s on a modern laptop.

## 1. Introduction

### Reference turbine

The *IEA-15-240-RWT* (Gaertner et al. 2020, NREL/TP-5000-75698) on
the UMaine VolturnUS-S semi-submersible platform (Allen et al. 2020,
NREL/TP-5000-76773) is the canonical 15 MW floating benchmark across
the IEA Wind community. Its WindIO ontology and matching OpenFAST
decks ship in the upstream `IEA-15-240-RWT` GitHub repository.

### Prerequisites

This notebook reads the WindIO ontology
`docs/OpenFAST_files/IEA-15-240-RWT/WT_Ontology/IEA-15-240-RWT_VolturnUS-S.yaml`
plus the companion OpenFAST decks under the same turbine tree. They
are gitignored (see `CLAUDE.md` *Independence stance*), so a fresh
clone won't run this notebook without first checking out the upstream
repo under `docs/OpenFAST_files/`. The `[windio]` extra (PyYAML) is
required: `pip install -e ".[dev,plots,windio]"`.

### Expected outputs

- Composite-blade and tubular-tower parked frequencies + mode-shape
  plots.
- The screening-tier `UserWarning` and the industry-grade coupled
  platform rigid-body + tower frequencies (auto-labelled DOFs).
- A `pybmodes windio` report written to a temp directory.
- A Campbell diagram and a WindIO-vs-ElastoDyn tower MAC heatmap.
- A consolidated frequency table.

## 2. Imports, path setup, and companion-deck discovery

The discovery call is the *same* scoped resolver `pybmodes windio`
uses internally: from the ontology it walks up to the turbine root
(the nearest ancestor with an `OpenFAST/` child) and picks the
configuration-matched HydroDyn/MoorDyn/ElastoDyn decks — it never
recursively scans an arbitrary parent and never grabs another
turbine's decks.

In [ ]:
import pathlib
import sys
import tempfile
import warnings

# Make pybmodes importable from a source checkout where the package
# isn't pip-installed (the project's default dev setup; see CLAUDE.md
# "Running tests on this machine").
_cwd = pathlib.Path.cwd().resolve()
_repo = next(
    (p for p in (_cwd, *_cwd.parents) if (p / "src" / "pybmodes").is_dir()),
    _cwd,
)
_src = _repo / "src"
if _src.is_dir() and str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

import numpy as np
import matplotlib.pyplot as plt

from pybmodes.plots import apply_style, plot_mode_shapes, bir_mode_shape_plot
from pybmodes.mac import compare_modes, plot_mac
from pybmodes.campbell import campbell_sweep, plot_campbell
from pybmodes.models import RotatingBlade, Tower
from pybmodes.cli import _discover_windio_inputs, main

apply_style()

REPO_ROOT = _repo
YAML = (
    REPO_ROOT / "docs" / "OpenFAST_files" / "IEA-15-240-RWT"
    / "WT_Ontology" / "IEA-15-240-RWT_VolturnUS-S.yaml"
)
if not YAML.is_file():
    raise FileNotFoundError(
        f"WindIO ontology not found at {YAML}. Clone the upstream "
        "IEA-15-240-RWT GitHub repo into docs/OpenFAST_files/ to run "
        "this notebook (see CLAUDE.md 'Independence stance')."
    )

disc = _discover_windio_inputs(YAML)
print(f"ontology : {YAML.name}")
for k in ("hydrodyn", "moordyn", "elastodyn"):
    tag = disc[k].name if disc[k] else "— (none → screening preview)"
    print(f"  {k:9s}: {tag}")
ELASTODYN = disc["elastodyn"]

## 3. Composite-layup blade — `RotatingBlade.from_windio`

The blade is **not** read from a deck shortcut: its composite layup
(spar caps, shell skins, shear webs, the airfoil `nd_arc` spine) is
reduced to distributed `mass / EA / EI / GJ` by a PreComp-class
thin-wall multi-cell Bredt–Batho classical-lamination analysis
(`pybmodes.io._precomp` + `pybmodes.io.windio_blade`). Mass and EA
are PreComp-class against the turbine's own BeamDyn 6×6; GJ/EI are
diagonal-reduction approximate (a documented limitation — see
`VALIDATION.md`).

In [ ]:
blade = RotatingBlade.from_windio(YAML)
bl = blade.run(n_modes=8, check_model=False)
print("Composite-layup blade — first 6 parked frequencies (Hz):")
for i in range(6):
    print(f"  mode {i + 1}: {float(bl.frequencies[i]):8.4f}")

In [ ]:
fig = plot_mode_shapes(
    bl, n_modes=4, component="both",
    title="IEA-15 composite blade — first 4 mode shapes (from the WindIO layup)",
)
fig.set_size_inches(13, 4)
plt.show()

## 4. Tubular tower — `Tower.from_windio`

The tower comes straight from the ontology geometry (outer diameter,
summed layer wall thickness, isotropic steel, `outfitting_factor`,
reference-axis span) through the closed-form circular-tube reduction.
For IEA-15 this is the path that reproduces the WISDEM-generated
ElastoDyn tower table to machine precision (see
`cases/ECOSYSTEM_FINDING.md`).

In [ ]:
tower = Tower.from_windio(YAML)
tw = tower.run(n_modes=8, check_model=False)
print("Tubular tower (cantilever, from ontology geometry) — first 4 (Hz):")
for i in range(4):
    print(f"  mode {i + 1}: {float(tw.frequencies[i]):8.4f}")

In [ ]:
fig = bir_mode_shape_plot(
    tw,
    [(1, "flap", "mode 1"), (2, "flap", "mode 2"),
     (3, "flap", "mode 3"), (4, "flap", "mode 4")],
    title="IEA-15 WindIO tower — fore-aft mode shapes (Bir 2010 convention)",
)
plt.show()

## 5. Coupled FOWT — two-tier fidelity

`Tower.from_windio_floating` has two fidelity tiers by design:

- **Screening preview (yaml only).** Member-Morison hydro + catenary
  mooring estimated from the ontology alone. Useful for early-stage
  screening; it emits one `UserWarning` explicitly naming itself
  `SCREENING-fidelity (NOT industry-grade)` so the fidelity is never
  silent. Morison ≠ BEM (heave especially), and the structural mass
  is a deliberate lower bound (trim ballast excluded).
- **Industry-grade (companion decks present).** With the discovered
  HydroDyn/MoorDyn/ElastoDyn decks it builds the full deck-backed
  coupled model — byte-identical to the BModes-JJ-validated
  `from_elastodyn_with_mooring` path except the tower is the
  machine-exact WindIO one. All six platform rigid-body modes + 1st
  tower bending land within 0.0–0.3 % of that reference.

In [ ]:
# --- screening tier: yaml only, no companion decks ---------------
with warnings.catch_warnings(record=True) as wlog:
    warnings.simplefilter("always")
    fowt_screen = Tower.from_windio_floating(YAML, water_depth=200.0)
res_screen = fowt_screen.run(n_modes=12, check_model=False)

screen_warns = [str(w.message) for w in wlog
                if "SCREENING-fidelity" in str(w.message)]
print("screening-tier UserWarning(s) — the fidelity is never silent:")
for m in screen_warns:
    print(f"  ! {m}")
print()
print("screening-tier first 8 frequencies (Hz):")
print("  " + ", ".join(f"{float(x):.4f}" for x in res_screen.frequencies[:8]))

In [ ]:
# --- industry-grade tier: companion decks present ----------------
fowt = Tower.from_windio_floating(
    YAML,
    hydrodyn_dat=disc["hydrodyn"],
    moordyn_dat=disc["moordyn"],
    elastodyn_dat=disc["elastodyn"],
)
res = fowt.run(n_modes=12, check_model=False)

# pyBmodes auto-classifies the six platform rigid-body modes for a
# free-free floating model; flexible tower modes stay None. Read the
# labels off mode_labels rather than assuming a DOF order — IEA-15
# UMaine's modal order is NOT the textbook surge/sway/heave/...
labels = res.mode_labels or [None] * 12
print("industry-grade (deck-backed) coupled solve:")
print(f"  {'mode':<6}{'label':<16}{'Hz':>10}{'period s':>11}")
for i in range(8):
    name = labels[i] if labels[i] is not None else "(elastic/coupled)"
    f = float(res.frequencies[i])
    per = 1.0 / f if f > 1e-9 else float("inf")
    print(f"  {i + 1:<6}{name:<16}{f:>10.4f}{per:>11.1f}")

## 6. The one-click orchestrator — `pybmodes windio`

Everything above in a single command. The orchestrator discovers the
ontology + companion decks, solves blade + tower + coupled platform,
optionally sweeps a Campbell diagram, and writes a bundled report.
Because the IEA-15 VolturnUS-S tree carries the companion decks, the
floating platform is industry-grade and **no** screening warning is
emitted (contrast § 5's yaml-only call).

In [ ]:
out_dir = pathlib.Path(tempfile.mkdtemp(prefix="windio_walkthrough_"))
report = out_dir / "iea15_volturnus_windio_report.md"
rc = main([
    "windio", str(YAML),
    "--out", str(report),
    "--n-modes", "12",
    "--campbell", "--max-rpm", "8.0", "--n-steps", "9",
])
print(f"\n`pybmodes windio` exit code: {rc}  (0 = OK)")
print(f"report written: {report}  ({report.stat().st_size} bytes)")
print("\n--- report head ---")
print("\n".join(report.read_text(encoding="utf-8").splitlines()[:28]))

## 7. Campbell diagram

The Campbell sweep rotates the blade through the operating rotor-speed
range and tracks each mode under centrifugal stiffening; tower modes
overlay as horizontal lines. IEA-15 rated rotor speed is 7.56 rpm
(Gaertner 2020 §3.6); the classic check is whether any *N* P
excitation line crosses an elastic mode near rated. The one-click
command above already wrote a Campbell PNG/CSV next to the report;
here we render it inline from the same validated sweep.

In [ ]:
if ELASTODYN is None:
    print("no companion ElastoDyn deck discovered — Campbell skipped")
else:
    omega_rpm = np.linspace(0.0, 8.0, 17)
    camp = campbell_sweep(
        ELASTODYN, omega_rpm=omega_rpm,
        n_blade_modes=4, n_tower_modes=2,
    )
    print(f"Campbell: {camp.frequencies.shape} (n_steps, n_modes)")
    print(f"labels: {camp.labels}")
    fig = plot_campbell(camp, rated_rpm=7.56, excitation_orders=[1, 3, 6])
    fig.set_size_inches(8, 5)
    plt.show()

## 8. MAC — screening vs industry-grade platform

Both floating solves in § 5 use the *same* machine-exact WindIO tower
mesh; only the platform tier differs (yaml-only Morison + catenary
screening vs the deck-backed coupled model). The Modal Assurance
Criterion quantifies how much swapping the screening platform for the
industry-grade one rotates the coupled mode shapes, and the per-pair
frequency shift *is* the screening error the § 5 `UserWarning` flags —
largest on heave (Morison ≠ BEM) and on the mooring-set surge/sway.

In [ ]:
cmp = compare_modes(
    res_screen, res,
    label_A="screening platform", label_B="industry-grade platform",
)
print("Hungarian-paired modes (screening -> industry-grade), % freq shift:")
sl = res_screen.mode_labels or [None] * 12
for k, (i, j) in enumerate(cmp.paired_modes[:8]):
    tag = sl[i] if i < len(sl) and sl[i] is not None else f"mode {i + 1}"
    print(f"  {tag:<16} {cmp.frequency_shift[k]:+8.1f} %")
fig = plot_mac(cmp)
fig.set_size_inches(6, 5)
plt.show()

## 9. Summary table

Every frequency this notebook computed, in one place. The coupled
platform rigid-body labels come straight from the pyBmodes classifier
(`res.mode_labels`) in modal order — no hand-typed DOF-order
assumption.

In [ ]:
rows: list[tuple[str, str]] = []
labels = res.mode_labels or [None] * 12
for i in range(6):
    name = labels[i] if labels[i] is not None else f"mode {i + 1} (coupled)"
    rows.append((f"platform {name}", f"{float(res.frequencies[i]):.4f} Hz"))
rows.append(("", ""))
rows.append(("1st tower bend (coupled)", f"{float(res.frequencies[6]):.4f} Hz"))
rows.append(("2nd tower bend (coupled)", f"{float(res.frequencies[7]):.4f} Hz"))
rows.append(("1st tower (WindIO cantilever)", f"{float(tw.frequencies[0]):.4f} Hz"))
rows.append(("2nd tower (WindIO cantilever)", f"{float(tw.frequencies[1]):.4f} Hz"))
rows.append(("", ""))
for i in range(4):
    rows.append((f"blade mode {i + 1} (parked)", f"{float(bl.frequencies[i]):.4f} Hz"))

print(f"{'mode':<34}{'frequency':>14}")
print("-" * 48)
for label, value in rows:
    print(f"{label:<34}{value:>14}")

## 10. Key findings

- **One ontology file, the whole turbine.** Blade (composite-layup
  reduction), tower (closed-form tube), and the coupled floating
  platform all come from a single WindIO `.yaml` — the 1.4.0 issue #35
  deliverable.
- **Fidelity is never silent.** The yaml-only floating tier in § 5
  emits an explicit `SCREENING-fidelity (NOT industry-grade)`
  `UserWarning`; supplying (or auto-discovering) the companion decks
  promotes it to the BModes-JJ-validated coupled model with no
  warning, as in § 6.
- **The one-click orchestrator reproduces the manual path.** § 6's
  `pybmodes windio` exit code 0 and report match the constructor
  results in §§ 3–5 — the CLI is a thin wrapper over the same
  validated solves.
- **The two tiers describe the same modes, at shifted frequencies.**
  The § 8 MAC stays diagonally dominant (same WindIO tower mesh, same
  mode topology) while the per-pair frequency shift is large on heave
  and the mooring-set DOFs — a quantitative picture of exactly the
  screening error the § 5 warning flags, and why the deck-backed tier
  exists.
- **Polynomial story unchanged.** This notebook reports *coupled
  frequencies*; ElastoDyn polynomial-coefficient generation still
  uses the cantilever `Tower.from_elastodyn` basis for floating decks
  — see `cases/ECOSYSTEM_FINDING.md` and
  `src/pybmodes/_examples/reference_decks/FLOATING_CASES.md`.